### Beyond the Squeaky Wheel: 311 Engagement & Equity Analysis
### Notebook 7: Engagement Score Construction

Calculates the Engagement Score as the gap between the 311 Service Request Index (SRI) and the Service Need Index (SNI) for each tract. Exports the merged tract-level scores, city-level summary tables, and comparison histograms across score transformations.

In [ ]:
# Step 1: Import libraries

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm

In [ ]:
# Step 2: Set up file paths and load inputs

# Sheet 1: SNI (Service Need Index) scores
SNI_PATH = "INSERT FILE PATH: SNI tract scores CSV"

# Sheet 2: 311 engagement index scores
SCORE_311_PATH = "INSERT FILE PATH: 311 SRI tract scores CSV"

# Output: joined data + Over-Under Engagement Score
OUTPUT_PATH = "INSERT FILE PATH: engagement score output CSV"
SUMMARY_PATH = "INSERT FILE PATH: city-level engagement score summary CSV"
SIGN_SUMMARY_PATH = "INSERT FILE PATH: city-level positive/negative count summary CSV"

# Column(s) shared between both sheets to join on (e.g. tract GEOID, city name)
JOIN_KEYS = ["GEOID"]

PDF_OUTPUT_PATH = "INSERT FILE PATH: output histogram PDF"

In [ ]:
# Sanity Check: confirm input files exist

print(f"SNI file found: {os.path.exists(SNI_PATH)}")
print(f"311 file found: {os.path.exists(SCORE_311_PATH)}")

In [ ]:
# Step 3: Load, join, and calculate Engagement Score

# GEOID must be read as string to preserve leading zeros (tracts in states 01-09)
dtype_overrides = {key: str for key in JOIN_KEYS}

sni_df = pd.read_csv(SNI_PATH, dtype=dtype_overrides)
score_311_df = pd.read_csv(SCORE_311_PATH, dtype=dtype_overrides)

# Zero-pad GEOID to 11 digits if present among join keys
for key in JOIN_KEYS:
    if "GEOID" in key.upper():
        sni_df[key] = sni_df[key].str.zfill(11)
        score_311_df[key] = score_311_df[key].str.zfill(11)

# Outer join keeps every row from both sheets, matching on JOIN_KEYS
merged_df = pd.merge(
    sni_df,
    score_311_df,
    on=JOIN_KEYS,
    how="outer",
)

# Over-Under Engagement Score: SNI_score minus 311_score
        # Observed minus Expected
merged_df["ES_raw"] = merged_df["311_score_raw"] - merged_df["SNI_Score"]
merged_df["ES_log"] = merged_df["311_score_log"] - merged_df["SNI_Score"]
merged_df["ES_sqrt"] = merged_df["311_score_sqrt"] - merged_df["SNI_Score"]
merged_df["ES_yj"] = merged_df["311_score_yj"] - merged_df["SNI_Score"]

tqdm.write(f"Merged rows: {len(merged_df)}")
merged_df.head()

In [ ]:
# Step 4: Export merged and export Engagement Score

merged_df.to_csv(OUTPUT_PATH, index=False)
tqdm.write(f"Exported to: {OUTPUT_PATH}")

In [ ]:
# Step 5: Generate summary table of basic stats by city

score_cols = ["ES_raw", "ES_log", "ES_sqrt", "ES_yj"]
stats = ["count", "mean", "median", "min", "max"]

city_col = next((c for c in merged_df.columns if c.lower() == "city_sni"), None)

if city_col:
    summary_df = merged_df.groupby(city_col)[score_cols].agg(stats)
    summary_df.columns = ["_".join(col) for col in summary_df.columns]
    summary_df = summary_df.reset_index()
else:
    summary_df = merged_df[score_cols].agg(stats).T
    summary_df.columns = stats
    summary_df = summary_df.reset_index().rename(columns={"index": "score_column"})

summary_df

summary_df.to_csv(SUMMARY_PATH, index=False)
tqdm.write(f"Exported to: {SUMMARY_PATH}")

In [ ]:
# Step 6: Configure engagement score column names

# EDIT: confirm these match your actual engagement column names
ENGAGEMENT_COLS = {
    "raw": "ES_raw",
    "log": "ES_log",
    "sqrt": "ES_sqrt",
    "yj": "ES_yj",
}

CITY_COL = "city_sni"

In [ ]:
# Step 7: Generate 2x2 histograms per city and export PDF
cities = sorted(merged_df[CITY_COL].dropna().unique())

with PdfPages(PDF_OUTPUT_PATH) as pdf:
    for city in cities:
        subset = merged_df[merged_df[CITY_COL] == city]

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for ax, (label, col) in zip(axes, ENGAGEMENT_COLS.items()):
            ax.hist(subset[col].dropna(), bins=30, color="steelblue", edgecolor="white")
            ax.set_title(f"{city} - {label}")

        plt.suptitle(city, fontsize=14)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved engagement score histograms to {PDF_OUTPUT_PATH}")

In [ ]:
# Step 8: Generate summary table of positive/negative counts by city

score_labels = {
    "ES_raw": "Raw",
    "ES_log": "Log",
    "ES_sqrt": "Sqrt",
    "ES_yj": "Yj"
}

def sign_counts(df):
    row = {"total_tracts": len(df)}
    for col, label in score_labels.items():
        row[f"{label}_negative"] = (df[col] < 0).sum()
        row[f"{label}_positive"] = (df[col] > 0).sum()
    return pd.Series(row)

if city_col:
    sign_summary_df = merged_df.groupby(city_col).apply(sign_counts).reset_index()
else:
    sign_summary_df = sign_counts(merged_df).to_frame().T

sign_summary_df

sign_summary_df.to_csv(SIGN_SUMMARY_PATH, index=False)
tqdm.write(f"Exported to: {SIGN_SUMMARY_PATH}")